# Setup

In [ ]:
import os
import requests

# DeepSeek API 配置
DEEPSEEK_API_KEY = "sk-7aac33072d424a95a52fbb9fda6a2dcd"
API_URL = "https://api.deepseek.com/v1/chat/completions"

SYSTEM_PROMPT = (
    "You are a precise agent. Your job is to generate the next Thought and Action in a ReAct loop. "
    "You MUST output exactly one Thought and one Action line in this format:\n"
    "Thought N: <your reasoning>\n"
    "Action N: <action>\n\n"
    "Rules:\n"
    "- NEVER generate Observation lines — the system will provide them.\n"
    "- Valid actions: Search[entity], Lookup[keyword], Finish[answer], Think[thought].\n"
    "- Action names MUST start with an uppercase letter.\n"
    "- Output ONLY the Thought and Action, nothing else."
)

def llm(prompt, stop=["\n"]):
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "deepseek-chat",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0,
        "max_tokens": 80,
        "top_p": 1,
        "stop": stop
    }
    resp = requests.post(API_URL, headers=headers, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

In [4]:
import wikienv, wrappers
env = wikienv.WikiEnv()
env = wrappers.HotPotQAWrapper(env, split="dev")
env = wrappers.LoggingWrapper(env)

def step(env, action):
    attempts = 0
    while attempts < 10:
        try:
            return env.step(action)
        except requests.exceptions.Timeout:
            attempts += 1

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


# ReAct

In [ ]:
import json
import re
import sys

folder = './prompts/'
prompt_file = 'prompts_naive.json'
with open(folder + prompt_file, 'r') as f:
    prompt_dict = json.load(f)

webthink_examples = prompt_dict['webthink_simple6']
instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types: 
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
Here are some examples.
"""
webthink_prompt = instruction + webthink_examples


def parse_thought_action(raw_text):
    """Robustly parse Thought and Action from LLM output."""
    text = raw_text.strip()

    # Pattern 1: "Thought N: ...\nAction N: ..."
    m = re.search(r'^(Thought \d+:.*?)\n(Action \d+:.*?)$', text, re.DOTALL)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    # Pattern 2: Thought on one line, Action on next (missing Action number)
    m = re.search(r'^(Thought \d+:.*?)\n(Action:.*?)$', text, re.DOTALL)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    # Pattern 3: Only Thought found, Action is just "Action" with no content
    m = re.search(r'^(Thought \d+:.*?)\n(Action.*?)$', text, re.DOTALL)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    # Fallback: take first line as Thought, rest as Action
    lines = text.split('\n')
    if len(lines) >= 2:
        thought = lines[0].strip()
        action = '\n'.join(lines[1:]).strip()
        return thought, action

    return None, None


def fix_action(action_text, step_num):
    """Ensure action starts with a valid command."""
    # Strip any stray "Action N:" prefix
    action_text = re.sub(r'^Action\s*\d*\s*:\s*', '', action_text).strip()

    # Check if it's a valid action format
    valid_actions = ['Search[', 'Lookup[', 'Finish[', 'Think[',
                     'search[', 'lookup[', 'finish[', 'think[']
    if any(action_text.startswith(a) for a in valid_actions):
        return action_text

    # Might be a thought/observation hallucination, try to extract an action
    m = re.search(r'(Search\[.+?\]|Lookup\[.+?\]|Finish\[.+?\]|Think\[.+?\])', action_text)
    if m:
        return m.group(1)

    return None


def webthink(idx=None, prompt=webthink_prompt, to_print=True):
    question = env.reset(idx=idx)
    if to_print:
        print(idx, question)
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0
    for i in range(1, 8):
        n_calls += 1
        thought_action = llm(prompt + f"Thought {i}:", stop=[f"\nObservation {i}:"])
        thought, action = parse_thought_action(thought_action)

        if thought is None or action is None:
            print('ohh...', repr(thought_action))
            n_badcalls += 1
            n_calls += 1
            # Retry with explicit Action prefix
            thought = thought_action.strip().split('\n')[0] if thought is None else thought
            action = llm(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=[f"\n"]).strip()

        fixed_action = fix_action(action, i)
        if fixed_action is None:
            print('bad action...', repr(action))
            # Last resort: try to extract any valid action from the raw output
            m = re.search(r'(Search\[.+?\]|Lookup\[.+?\]|Finish\[.+?\]|Think\[.+?\])', thought_action)
            if m:
                fixed_action = m.group(1)
            else:
                fixed_action = "Think[no valid action found]"

        obs, r, done, info = step(env, fixed_action[0].lower() + fixed_action[1:])
        obs = obs.replace('\\n', '')
        step_str = f"Thought {i}: {thought}\nAction {i}: {fixed_action}\nObservation {i}: {obs}\n"
        prompt += step_str
        if to_print:
            print(step_str)
        if done:
            break
    if not done:
        obs, r, done, info = step(env, "finish[]")
    if to_print:
        print(info, '\n')
    info.update({'n_calls': n_calls, 'n_badcalls': n_badcalls, 'traj': prompt})
    return r, info

In [6]:
import random
import time
idxs = list(range(7405))
random.Random(233).shuffle(idxs)

rs = []
infos = []
old_time = time.time()
for i in idxs[:500]:
    r, info = webthink(i, to_print=True)
    rs.append(info['em'])
    infos.append(info)
    print(sum(rs), len(rs), sum(rs) / len(rs), (time.time() - old_time) / len(rs))
    print('-----------')
    print()

3687 Question: What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?
Thought 1: Thought 1: I need to identify the American action crime thriller film directed by Stuart Bird, then find which movie actress Irene Jacob completed before that film.

Action 1: Search[Stuart Bird]
Observation 1: 

Thought 2: Observation 1: The search for "Stuart Bird" did not return a direct result. It might be a misspelling or less known entity. I should try searching for "American action crime thriller film directed by Stuart Bird" or check if "Stuart Bird" is a director with a specific film.

Action 2: Search[American action crime thriller film Stuart Bird]
Observation 2: 

Thought 3: Observation 2: The search did not return any relevant results. It is possible that "Stuart Bird" is a misspelling or a less common name. I should try searching for "Stuart Bird director" to see if there is a known film director by that name.

Action 3: Search[Stua

KeyboardInterrupt: 